# 🎓 AutoAttendance System - Complete Implementation
## All-in-One Jupyter Notebook

This notebook contains the complete AutoAttendance face recognition system:
- ✅ Data Collection from Webcam
- ✅ Training with InsightFace Embeddings
- ✅ Real-time Face Recognition
- ✅ Attendance Recording

**Workflow:**
1. **Section 1:** Setup & Dependencies
2. **Section 2:** Collect Face Data
3. **Section 3:** Train Model (Generate Embeddings)
4. **Section 4:** Run Recognition & Record Attendance

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'opencv-python',
    'numpy',
    'pandas',
    'openpyxl',
    'insightface',
    'onnxruntime',
    'scikit-image',
    'python-dotenv'
]

print("Installing required packages...")
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

print("\n✅ All packages ready!")

In [ ]:
# Imports and Setup
import os
import cv2
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime
from pathlib import Path
import pickle
import warnings
from IPython.display import display, Image, clear_output
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

warnings.filterwarnings('ignore')

# Import InsightFace
from insightface.app import FaceAnalysis

print("✅ All imports successful!")

In [ ]:
# Configuration and Directory Setup
PROJECT_ROOT = Path('h:/AutoAttendance')
DATA_DIR = PROJECT_ROOT / 'data'
FACES_DIR = DATA_DIR / 'faces'
TRAINING_DIR = DATA_DIR / 'training'
ATTENDANCE_DIR = DATA_DIR / 'attendance'
MODELS_DIR = PROJECT_ROOT / 'models'
DATABASE_PATH = MODELS_DIR / 'attendance.sqlite3'

# Create directories
for dir_path in [FACES_DIR, TRAINING_DIR, ATTENDANCE_DIR, MODELS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Configuration parameters
CONFIG = {
    'INSIGHTFACE_MODEL': 'buffalo_l',
    'INSIGHTFACE_DET_SIZE': (640, 640),
    'RECOGNITION_THRESHOLD': 0.45,  # Cosine distance threshold
    'CAMERA_ID': 0,
    'FRAME_WIDTH': 640,
    'FRAME_HEIGHT': 480,
    'SAMPLES_PER_PERSON': 80,
}

print(f"✅ Directories created at: {PROJECT_ROOT}")
print(f"✅ Config loaded: {CONFIG}")
print(f"✅ Database path: {DATABASE_PATH}")

In [ ]:
# Database Module - Manage Embeddings
class AttendanceDatabase:
    def __init__(self, db_path):
        self.db_path = db_path
        self._init_db()
    
    def _init_db(self):
        """Initialize SQLite database with required tables."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Students table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS students (
                student_id INTEGER PRIMARY KEY AUTOINCREMENT,
                student_name TEXT UNIQUE NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        
        # Face embeddings table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS face_embeddings (
                embedding_id INTEGER PRIMARY KEY AUTOINCREMENT,
                student_id INTEGER NOT NULL,
                embedding BLOB NOT NULL,
                quality_score REAL,
                model_name TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(student_id) REFERENCES students(student_id)
            )
        ''')
        
        # Attendance table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS attendance_log (
                log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                student_id INTEGER NOT NULL,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                confidence REAL,
                FOREIGN KEY(student_id) REFERENCES students(student_id)
            )
        ''')
        
        conn.commit()
        conn.close()
        print("✅ Database initialized")
    
    def add_student(self, name):
        """Add or get student ID."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('INSERT OR IGNORE INTO students (student_name) VALUES (?)', (name,))
        cursor.execute('SELECT student_id FROM students WHERE student_name = ?', (name,))
        student_id = cursor.fetchone()[0]
        conn.commit()
        conn.close()
        return student_id
    
    def add_embedding(self, student_id, embedding, quality_score=0):
        """Store embedding in database."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        embedding_blob = pickle.dumps(embedding)
        cursor.execute('''
            INSERT INTO face_embeddings (student_id, embedding, quality_score, model_name)
            VALUES (?, ?, ?, ?)
        ''', (student_id, embedding_blob, quality_score, CONFIG['INSIGHTFACE_MODEL']))
        conn.commit()
        conn.close()
    
    def load_embeddings(self):
        """Load all embeddings from database."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            SELECT e.embedding, s.student_id, s.student_name
            FROM face_embeddings e
            JOIN students s ON e.student_id = s.student_id
        ''')
        results = []
        for row in cursor.fetchall():
            embedding = pickle.loads(row[0])
            results.append({
                'embedding': embedding,
                'student_id': row[1],
                'student_name': row[2]
            })
        conn.close()
        return results
    
    def log_attendance(self, student_id, confidence):
        """Record attendance."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO attendance_log (student_id, confidence)
            VALUES (?, ?)
        ''', (student_id, confidence))
        conn.commit()
        conn.close()
    
    def get_attendance_today(self):
        """Get today's attendance."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        today = datetime.now().date()
        cursor.execute('''
            SELECT s.student_name, a.timestamp
            FROM attendance_log a
            JOIN students s ON a.student_id = s.student_id
            WHERE DATE(a.timestamp) = ?
            ORDER BY a.timestamp
        ''', (today,))
        results = cursor.fetchall()
        conn.close()
        return results

# Initialize database
db = AttendanceDatabase(str(DATABASE_PATH))
print("✅ Database module ready")

In [ ]:
# InsightFace Recognition Module
class FaceRecognitionEngine:
    def __init__(self):
        print("Loading InsightFace model...")
        self.app = FaceAnalysis(name=CONFIG['INSIGHTFACE_MODEL'])
        self.app.prepare(ctx_id=-1, det_size=CONFIG['INSIGHTFACE_DET_SIZE'])
        self.known_embeddings = []
        self.reverse_labels = {}
        print("✅ InsightFace model loaded")
    
    def detect_faces(self, frame):
        """Detect faces and get embeddings."""
        try:
            faces = self.app.get(frame)
            return faces
        except Exception as e:
            print(f"Error detecting faces: {e}")
            return []
    
    def get_embedding(self, face):
        """Extract embedding from detected face."""
        return face.embedding
    
    def load_known_embeddings(self):
        """Load all registered embeddings from database."""
        self.known_embeddings = db.load_embeddings()
        self.reverse_labels = {
            item['student_id']: item['student_name'] 
            for item in self.known_embeddings
        }
        print(f"✅ Loaded {len(self.known_embeddings)} embeddings")
    
    def cosine_distance(self, embedding1, embedding2):
        """Calculate cosine distance between two embeddings."""
        # Normalize embeddings
        norm1 = np.linalg.norm(embedding1)
        norm2 = np.linalg.norm(embedding2)
        
        if norm1 == 0 or norm2 == 0:
            return 1.0
        
        embedding1 = embedding1 / norm1
        embedding2 = embedding2 / norm2
        
        # Cosine distance = 1 - cosine_similarity
        return 1 - np.dot(embedding1, embedding2)
    
    def recognize_face(self, embedding):
        """Match embedding against known embeddings."""
        if not self.known_embeddings:
            return None, None, 1.0
        
        best_match = None
        best_distance = float('inf')
        
        for known in self.known_embeddings:
            distance = self.cosine_distance(embedding, known['embedding'])
            if distance < best_distance:
                best_distance = distance
                best_match = known
        
        # Check if match is within threshold
        if best_distance < CONFIG['RECOGNITION_THRESHOLD']:
            return best_match['student_name'], best_match['student_id'], best_distance
        else:
            return None, None, best_distance

# Initialize recognition engine
recognizer = FaceRecognitionEngine()
print("✅ Recognition engine ready")

---

# 📸 SECTION 1: COLLECT FACE DATA

**What to do:**
1. Run the cell below
2. Enter a person's name when prompted
3. Position face in front of camera
4. Press 'c' to capture (collect 80 samples)
5. Press 'q' to finish and move to next person
6. Repeat for each person you want to register

**Tips:**
- Capture faces at different angles
- Vary lighting conditions
- Make sure face is clearly visible
- 80+ samples = best results

In [ ]:
# Data Collection Function
def collect_face_data():
    """Collect face samples from webcam."""
    print("\n" + "="*60)
    print("FACE DATA COLLECTION")
    print("="*60)
    
    person_name = input("\nEnter person's name (or 'done' to finish): ").strip()
    
    if person_name.lower() == 'done':
        print("✅ Data collection finished!")
        return False
    
    if not person_name:
        print("❌ Name cannot be empty")
        return True
    
    person_dir = FACES_DIR / person_name
    person_dir.mkdir(parents=True, exist_ok=True)
    
    # Check existing samples
    existing = len(list(person_dir.glob('*.jpg')))
    print(f"\n📁 Storing images in: {person_dir}")
    print(f"📊 Existing samples: {existing}")
    print(f"🎯 Target: {CONFIG['SAMPLES_PER_PERSON']} samples")
    
    cap = cv2.VideoCapture(CONFIG['CAMERA_ID'])
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CONFIG['FRAME_WIDTH'])
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CONFIG['FRAME_HEIGHT'])
    
    sample_count = existing
    print(f"\n📷 Press 'c' to capture, 'q' to quit")
    
    while sample_count < CONFIG['SAMPLES_PER_PERSON']:
        ret, frame = cap.read()
        
        if not ret:
            print("❌ Cannot read from camera")
            break
        
        # Display frame with counter
        display_frame = frame.copy()
        cv2.putText(display_frame, f"Sample: {sample_count}/{CONFIG['SAMPLES_PER_PERSON']}", 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(display_frame, f"Person: {person_name}", 
                   (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
        cv2.putText(display_frame, "Press 'c' to capture, 'q' to quit", 
                   (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 1)
        
        cv2.imshow(f'Collecting data for {person_name}', display_frame)
        
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord('c'):
            # Save image
            filename = person_dir / f"{person_name}_{sample_count:03d}.jpg"
            cv2.imwrite(str(filename), frame)
            sample_count += 1
            print(f"✅ Saved sample {sample_count}/{CONFIG['SAMPLES_PER_PERSON']}")
        
        elif key == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    
    print(f"\n✅ Collected {sample_count - existing} new samples for {person_name}")
    print(f"📊 Total samples for {person_name}: {sample_count}")
    
    return True

# Run data collection
print("""
INSTRUCTIONS:
1. Run this cell
2. Enter a person's name
3. Capture 80 face samples
4. Repeat for all people
5. Type 'done' when finished

Press 'c' to capture, 'q' to finish with current person
""")

# Uncomment to run:
# while collect_face_data():
#     pass

---

# 🤖 SECTION 2: TRAIN MODEL (Generate Embeddings)

**What to do:**
1. Run the cell below after collecting face data
2. System will convert all face images → embeddings
3. Store embeddings in SQLite database
4. Takes 2-5 minutes

**How it works:**
- InsightFace reads each collected face image
- Generates a 512-dimensional embedding vector
- Stores in `models/attendance.sqlite3`
- These embeddings are used for recognition

In [ ]:
# Training Function - Generate Embeddings
def train_model():
    """Train model by generating embeddings for all collected faces."""
    print("\n" + "="*60)
    print("TRAINING MODEL - GENERATING EMBEDDINGS")
    print("="*60)
    
    # Find all person directories
    person_dirs = [d for d in FACES_DIR.iterdir() if d.is_dir()]
    
    if not person_dirs:
        print("❌ No face data found in data/faces/")
        return
    
    total_images = 0
    db.db  # Access database
    
    for person_dir in person_dirs:
        person_name = person_dir.name
        student_id = db.add_student(person_name)
        
        # Get all face images for this person
        images = list(person_dir.glob('*.jpg')) + list(person_dir.glob('*.png'))
        
        print(f"\n👤 Processing {person_name}...")
        print(f"   📷 Found {len(images)} images")
        
        for i, image_path in enumerate(images):
            try:
                # Read image
                image = cv2.imread(str(image_path))
                if image is None:
                    continue
                
                # Detect faces and get embeddings
                faces = recognizer.detect_faces(image)
                
                if len(faces) > 0:
                    # Use first detected face
                    face = faces[0]
                    embedding = recognizer.get_embedding(face)
                    quality_score = face.det_score * 100  # Convert to percentage
                    
                    # Store in database
                    db.add_embedding(student_id, embedding, quality_score)
                    
                    if (i + 1) % 10 == 0:
                        print(f"   ✅ Processed {i + 1}/{len(images)}")
                
            except Exception as e:
                print(f"   ⚠️  Error processing {image_path.name}: {e}")
        
        print(f"   ✅ Completed {person_name}: {len(images)} embeddings stored")
        total_images += len(images)
    
    # Load embeddings
    recognizer.load_known_embeddings()
    
    print(f"\n" + "="*60)
    print(f"✅ TRAINING COMPLETE!")
    print(f"   📊 Total embeddings: {total_images}")
    print(f"   👥 Total people: {len(person_dirs)}")
    print(f"   💾 Database: {DATABASE_PATH}")
    print("="*60)

# Run training
# Uncomment to train:
# train_model()

---

# ✅ SECTION 3: RUN REAL-TIME RECOGNITION & ATTENDANCE

**What to do:**
1. Make sure you've trained the model first
2. Run the cell below
3. System starts webcam and recognizes faces
4. Attendance is recorded in CSV + Database
5. Press 'q' to quit

**What happens:**
- Webcam starts streaming
- Detects faces in real-time
- Compares embeddings against database
- Shows recognized person's name
- Records in attendance log
- Prevents marking same person twice per session

In [ ]:
# Real-time Recognition & Attendance
def run_recognition():
    """Run real-time face recognition and record attendance."""
    print("\n" + "="*60)
    print("STARTING FACE RECOGNITION - ATTENDANCE SYSTEM")
    print("="*60)
    
    # Load embeddings from database
    recognizer.load_known_embeddings()
    
    if not recognizer.known_embeddings:
        print("❌ No trained embeddings found!")
        print("   Please run: train_model() first")
        return
    
    print(f"✅ Loaded {len(recognizer.known_embeddings)} embeddings")
    print(f"👥 Registered people: {len(set(e['student_name'] for e in recognizer.known_embeddings))}")
    
    # Track who was marked today
    marked_today = {}
    
    cap = cv2.VideoCapture(CONFIG['CAMERA_ID'])
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CONFIG['FRAME_WIDTH'])
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CONFIG['FRAME_HEIGHT'])
    
    print(f"\n📷 Starting webcam...")
    print(f"Press 'q' to quit, 's' to export attendance")
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            print("❌ Cannot read from camera")
            break
        
        frame_count += 1
        display_frame = frame.copy()
        
        # Detect faces
        faces = recognizer.detect_faces(frame)
        
        # Process each detected face
        for face in faces:
            try:
                # Get embedding for this face
                embedding = recognizer.get_embedding(face)
                
                # Recognize the face
                person_name, student_id, distance = recognizer.recognize_face(embedding)
                
                # Draw bounding box
                x1, y1, x2, y2 = map(int, face.bbox)
                
                if person_name:
                    # Known person
                    color = (0, 255, 0)  # Green
                    text = f"{person_name} ({distance:.2f})"
                    
                    # Record attendance (only once per session)
                    if person_name not in marked_today:
                        db.log_attendance(student_id, 1 - distance)
                        marked_today[person_name] = datetime.now()
                        print(f"✅ {person_name} - Attendance recorded at {marked_today[person_name].strftime('%H:%M:%S')}")
                    
                else:
                    # Unknown person
                    color = (0, 0, 255)  # Red
                    text = f"Unknown ({distance:.2f})"
                    print(f"⚠️  Unknown face detected")
                
                # Draw rectangle
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, 2)
                
                # Put text
                cv2.putText(display_frame, text, (x1, y1 - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                
            except Exception as e:
                print(f"Error processing face: {e}")
        
        # Display stats
        cv2.putText(display_frame, f"Marked: {len(marked_today)}", 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        cv2.putText(display_frame, "Press 'q' to quit, 's' to export", 
                   (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.imshow('AutoAttendance - Press q to quit', display_frame)
        
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord('q'):
            break
        elif key == ord('s'):
            export_attendance()
    
    cap.release()
    cv2.destroyAllWindows()
    
    print(f"\n" + "="*60)
    print("✅ RECOGNITION SESSION ENDED")
    print(f"📊 Total people marked: {len(marked_today)}")
    print(f"👥 People: {', '.join(marked_today.keys())}")
    print("="*60)
    
    return marked_today

# Run recognition
# Uncomment to start:
# marked = run_recognition()

In [ ]:
# Export & Reporting Functions
def export_attendance():
    """Export attendance to CSV."""
    today = datetime.now().date()
    results = db.get_attendance_today()
    
    if not results:
        print("❌ No attendance records today")
        return
    
    # Create DataFrame
    df = pd.DataFrame(results, columns=['Name', 'Time'])
    
    # Save to CSV
    csv_path = ATTENDANCE_DIR / f"attendance_{today}.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"\n✅ Attendance exported to: {csv_path}")
    print(df.to_string(index=False))
    
    return df

def get_attendance_summary():
    """Get attendance summary."""
    today = datetime.now().date()
    results = db.get_attendance_today()
    
    if not results:
        print("No attendance today")
        return pd.DataFrame()
    
    df = pd.DataFrame(results, columns=['Name', 'Time'])
    df['Date'] = today
    df['Time'] = pd.to_datetime(df['Time']).dt.strftime('%H:%M:%S')
    
    print(f"\n📊 Attendance Summary - {today}")
    print("="*50)
    print(df.to_string(index=False))
    print("="*50)
    
    return df

def show_registered_people():
    """Show all registered people."""
    registered = recognizer.reverse_labels
    
    if not registered:
        print("No people registered yet")
        return
    
    print("\n" + "="*50)
    print("📋 REGISTERED PEOPLE")
    print("="*50)
    for student_id, name in registered.items():
        count = len([e for e in recognizer.known_embeddings 
                    if e['student_id'] == student_id])
        print(f"  {student_id}. {name:20s} ({count} embeddings)")
    print("="*50)

# Uncomment to use:
# export_attendance()
# get_attendance_summary()
# show_registered_people()

In [ ]:
# Utility & Debugging Functions
def check_system_status():
    """Check system status and show statistics."""
    print("\n" + "="*60)
    print("🔍 SYSTEM STATUS CHECK")
    print("="*60)
    
    # Check directories
    print("\n📁 Directories:")
    for name, path in [('Faces', FACES_DIR), ('Models', MODELS_DIR), 
                       ('Attendance', ATTENDANCE_DIR)]:
        print(f"  {name:15s}: {path}")
    
    # Count collected images
    print("\n📸 Collected Images:")
    total_images = 0
    for person_dir in FACES_DIR.iterdir():
        if person_dir.is_dir():
            count = len(list(person_dir.glob('*.jpg'))) + len(list(person_dir.glob('*.png')))
            print(f"  {person_dir.name:20s}: {count}")
            total_images += count
    print(f"  {'Total':20s}: {total_images}")
    
    # Check database
    print("\n💾 Database:")
    print(f"  Path: {DATABASE_PATH}")
    print(f"  Size: {DATABASE_PATH.stat().st_size / 1024:.2f} KB")
    
    # Check embeddings
    print("\n🧠 Embeddings:")
    recognizer.load_known_embeddings()
    print(f"  Loaded: {len(recognizer.known_embeddings)}")
    print(f"  People: {len(set(e['student_name'] for e in recognizer.known_embeddings))}")
    
    # Check camera
    print("\n📷 Camera:")
    cap = cv2.VideoCapture(CONFIG['CAMERA_ID'])
    if cap.isOpened():
        print(f"  ✅ Camera is available")
        cap.release()
    else:
        print(f"  ❌ Camera not available")
    
    print("="*60)

def clear_database():
    """Clear all data (WARNING: destructive!)."""
    response = input("\n⚠️  WARNING: This will delete ALL data. Type 'yes' to confirm: ")
    if response.lower() == 'yes':
        import os
        if os.path.exists(DATABASE_PATH):
            os.remove(DATABASE_PATH)
        print("✅ Database cleared")
        db._init_db()
    else:
        print("❌ Cancelled")

def test_embedding_generation():
    """Test embedding generation on a sample face."""
    test_image = None
    
    # Find first available face image
    for person_dir in FACES_DIR.iterdir():
        if person_dir.is_dir():
            images = list(person_dir.glob('*.jpg'))
            if images:
                test_image = str(images[0])
                break
    
    if not test_image:
        print("❌ No test images found")
        return
    
    print(f"\n📸 Testing embedding generation...")
    print(f"   File: {test_image}")
    
    image = cv2.imread(test_image)
    faces = recognizer.detect_faces(image)
    
    if faces:
        embedding = recognizer.get_embedding(faces[0])
        print(f"   ✅ Embedding generated!")
        print(f"   📊 Size: {len(embedding)} dimensions")
        print(f"   📈 First 10 values: {embedding[:10]}")
    else:
        print(f"   ❌ No faces detected in test image")

# Show menu
def show_menu():
    """Show interactive menu."""
    print("""
╔════════════════════════════════════════════╗
║   AUTOATTENDANCE - JUPYTER NOTEBOOK       ║
║   Complete Face Recognition System        ║
╚════════════════════════════════════════════╝

MENU:
  1. check_system_status()    - Check everything
  2. collect_face_data()      - Collect faces (loop)
  3. train_model()            - Train (generate embeddings)
  4. run_recognition()        - Run attendance
  5. get_attendance_summary()  - Show today's attendance
  6. show_registered_people() - List all people
  7. export_attendance()       - Export to CSV
  8. test_embedding_generation() - Test system
  9. clear_database()         - DELETE ALL DATA

NEXT STEPS:
  ➡️  Run: check_system_status()
""")

show_menu()